<img src="http://www.cidaen.es/assets/img/mCIDaeNnb.png" alt="Logo CiDAEN" align="right">




<br><br><br>
<h2><font color="#00586D" size=4>Capstone II</font></h2>



<h1><font color="#00586D" size=5>Elaboración de una base de datos sobre películas y series</font></h1>

<br><br><br>


<div align="right">
<font color="#00586D" size=3>Luis de la Ossa</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube</font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>

</div>

---

En este capstone se aborda la primera parte del proceso del ciclo de datos: adquisición, almacenamiento y preparación. En el trabajo se recurrirá a fuentes de distinta naturaleza para la lectura de datos que, paralelamente, serán almacenados de manera provisional en formato semiestructurado. Después, esta información será procesada y almacenada en una base de datos relacional a partir de la cual se podrán hacer las consultas necesarias para llevar a cabo el análisis. Este ciclo, que reproduciremos en un ámbito "doméstico", tiene cierta similitud con el ciclo de ingeniería de datos en el que se adquieren datos, se almacenan en un *datalake* no estructurado, y a partir de ahí se extraen, preparan, y organizan de acuerdo al uso que se vaya a hacer de ellos. 
<br>

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Debido a al caracter didáctico del proyecto, se sacrifican ligeramente tanto el procecimiento de descarga (se introducen variaciones pero se podría hacer siempre igual) como la organización óptima de los datos. 
    
Por otra parte, debido al tamaño del proyecto y la estructura de índices y enlaces, es recomendable trabajar con `jupyter nbclassic`. 
</div>


---

<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Descarga y selección de datos de IMDb](#section1)
   * [1.1 Lectura de datos de IMDb](#section11)
   * [1.2 Selección de títulos](#section12) 
   
* [2. Obtención de datos de TMDb](#section2)
   * [2.1 Obtención de información sobre las películas](#section21)
   * [2.2 Almacenamiento en MongoDB](#section22)   
   * [2.3 Descarga de los créditos](#section23)  
   * [2.4 Datos de personas](#section24)  
   
* [3. Elaboración de un conjunto de datos estructurado](#section3)  
   * [3.1 Creación de DataFrames](#section31)   
   * [3.2 Creación de una base de datos relacional](#section32) 
   
* [4. Algunas consultas de ejemplo](#section4) 
* [5. Conclusiones](#section5) 

* [A. Trabajo adicional (opcional)](#sectionA) 
    * [A1. Palabras clave](#sectionA1)
    * [A2. Datos sobre series y episodios](#sectionA2)
---

<br>

In [1]:
# Opciones de visualización
from IPython.display import display, HTML, Image

display(HTML("<style>.container{ width:95%}</style>"))
%matplotlib inline
%config InlineBackend.figure_format = 'retina'


# Resuelve potenciales problemas de conexión con las APIs
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

<div align="right">
<a href="#indice"><font size=5 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---



<a id="section1"></a>
## <font color="#00586D"> 1. Descarga y selección de datos de IMDb</font>
<br>




_Internet Movie Database ([IMDb](http://www.imdb.com))_ es uno de los sitios de referencia sobre películas y series en internet. Entre otras cosas, almacena información sobre repartos, tramas, presupuestos, etc. Además, contiene  valoraciones y revisiones proporcionadas por los usuarios.  IMDb prohíbe expresamente la recolección de datos mediante scraping de su web a menos que se cuente con una autorización. Sin embargo,  pone a disposición pública algunos conjuntos de datos actualizados en formato `csv` [(enlace)](https://datasets.imdbws.com). A pesar de que la información que contienen estos archivos es reducida, en esta práctica la utilizaremos como punto de partida para la elaboración de un conjunto de datos más completo a partir de otras fuentes. En concreto, vamos a trabajar con la última versión (10 de diciembre de 2023) de dos archivos:

* `data/imdb/title.basics.tsv.gz`, que contiene información básica sobre los títulos existentes en los conjuntos de datos, y que pueden ser películas, series, episodios, etc. 
* `data/imdb/title.ratings.tsv.gz`, que contiene información sobre las valoraciones hechas a cada título.



<div align="right">
<a href="#indice"><font size=4 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>



<a id="section11"></a>
### <font color="#00586D"> 1.1 Lectura de datos de IMDb</font>



<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> 
Leer el contenido de los archivos y almacenarlo en los _DataFrames_ `df_titles` y `df_ratings`, respectivamente, **utilizando la primera columna como índice** en los dos casos. 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> Las columnas de estos archivos están separadas por tabulaciones (caracter `"\t"`), y los valores perdidos se representan con `'\N'`.  Por otra parte, el archivo `data/imdb/title.basics.tsv.gz` ocupa 197 MBytes. Sin embargo, al descomprimirlo ocupa 975 MBytes. La función `pd.read_csv()` de _Pandas_ también leer directamente archivos comprimidos directamente. Para ello, en este caso, hay que pasarle el parámetro `compression='gzip'` (la lectura requiere varios segundos).
</div>

En el caso de `data/imdb/title.basics.tsv.gz`, una inspección previa permite determinar que existen 4 columnas cuyos valores son enteros o nulos, por lo que es necesario especificar el tipo de datos correspondiente (`Int32`). En principio, esto se hará solamente para tres de ellas: `isAdult`, `startYear` y `endYear`.

In [ ]:
# Para poder mostrar los DataFrames enteros y ver el número de filas
import pandas as pd
pd.options.display.max_rows = 6
pd.options.mode.copy_on_write = True

num_types ={
    'titleType':'string',
    'primariTitle':'string',
    'originalTitle':'string',
    'genres':'string',
    'isAdult':'Int32',
    'startYear':'Int32',
    'endYear':'Int32',
}


df_titles = pd.read_csv('title.basics.tsv.gz',sep='\t',na_values='\\N',compression='gzip',dtype=num_types)
df_titles.head()


: 

La columna `runtimeMinutes` no se puede convertir directamente en el momento de la lectura porque, debido a un error, algunas filas contienen *strings*. En este caso, hay que proceder a posteriori en dos pasos: conversión a numérico mediante `pd.to_numeric`, de modo que los valores que no se puedan convertir se transformen en `np.NaN`; y conversión a `Int32` mediante `Series.astype()`.

In [4]:
df_titles['runtimeMinutes'] = pd.to_numeric(df_titles['runtimeMinutes'], errors='coerce').astype('Int32')
df_titles.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11217433 entries, 0 to 11217432
Data columns (total 9 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   tconst          object
 1   titleType       string
 2   primaryTitle    object
 3   originalTitle   string
 4   isAdult         Int32 
 5   startYear       Int32 
 6   endYear         Int32 
 7   runtimeMinutes  Int32 
 8   genres          string
dtypes: Int32(4), object(2), string(3)
memory usage: 641.9+ MB


In [5]:
df_titles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11217433 entries, 0 to 11217432
Data columns (total 9 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   tconst          object
 1   titleType       string
 2   primaryTitle    object
 3   originalTitle   string
 4   isAdult         Int32 
 5   startYear       Int32 
 6   endYear         Int32 
 7   runtimeMinutes  Int32 
 8   genres          string
dtypes: Int32(4), object(2), string(3)
memory usage: 641.9+ MB


<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Leer el contenido de `data/imdb/title.ratings.tsv.gz` y almacenarlo en el `DataFrame` `df_ratings`. 

In [6]:
df_ratings = pd.read_csv('title.ratings.tsv.gz',sep='\t',na_values='\\N',compression='gzip')
df_ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1495761 entries, 0 to 1495760
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   tconst         1495761 non-null  object 
 1   averageRating  1495761 non-null  float64
 2   numVotes       1495761 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 34.2+ MB


En este punto, se han cargado los 3 `DataFrames`, y todos están indexados mediante el campo `tconst`, que contiene un identificador único de cada recurso.

<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<div align="right">
<a href="#indice"><font size=4 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

<a id="section12"></a>
### <font color="#00586D"> 1.2 Selección de títulos</font>
<br>

Un vistazo a los datos obtenidos permite apreciar la gran cantidad de información que contienen los `DataFrame` (millones de filas). Además, se aprecia en `df_ratings` que algunas entradas tienen un número de votos extremadamente reducido. En esta parte se seleccionarán los títulos relevantes, que servirán como base para el resto del trabajo.

La siguiente celda muestra los tipos de títulos que contiene el `DataFrame` `df_titles`.

In [7]:
df_titles['titleType'].unique()

<StringArray>
[       'short',        'movie',      'tvShort',      'tvMovie',
    'tvEpisode',     'tvSeries', 'tvMiniSeries',    'tvSpecial',
        'video',    'videoGame',      'tvPilot']
Length: 11, dtype: string

#### <font color="#00586D"> Selección de películas</font>


<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Crear un `DataFrame` denominado `df_movies` que contenga todos los títulos de tipo `movie` de `df_titles`, además de su valoración media y número de votos. Considerar solamente aquellos títulos para los que el número de votos sea mayor (estrictamente) que 50000.



<br>

<details>    
<summary>
    <font size=3 color="#00586D"> <i class="fa fa-info-circle"  aria-hidden="true"></i> </font>
</summary>
   
* Se trata de hacer un `merge` entre `df_titles` y `df_ratings`.
* Es más eficiente seleccionar primero los títulos correspondientes a películas y las entradas de `df_ratings` con suficiente número de votaciones (con `DataFrame.query`, por ejemplo), y después hacer el `merge` (se puede hacer en la misma sentencia).
* Deben devolverse 4073 películas.

</details>   

In [8]:
df_titles.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,<NA>,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,<NA>,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,<NA>,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,<NA>,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,<NA>,1,"Comedy,Short"


In [9]:
df_ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2096
1,tt0000002,5.6,282
2,tt0000003,6.5,2115
3,tt0000004,5.4,182
4,tt0000005,6.2,2845


In [10]:
df_movies = (
    pd.merge(
        df_titles,
        df_ratings[df_ratings['numVotes']>50000],
        on = 'tconst',
        how='inner'
    )
)

In [27]:
df_movies.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0000417,short,A Trip to the Moon,Le voyage dans la lune,0,1902,<NA>,13,"Action,Adventure,Comedy",8.1,57200
1,tt0010323,movie,The Cabinet of Dr. Caligari,Das Cabinet des Dr. Caligari,0,1920,<NA>,67,"Horror,Mystery,Thriller",8.0,71863
2,tt0012349,movie,The Kid,The Kid,0,1921,<NA>,68,"Comedy,Drama,Family",8.2,137619
3,tt0013442,movie,Nosferatu: A Symphony of Horror,"Nosferatu, eine Symphonie des Grauens",0,1922,<NA>,94,"Fantasy,Horror",7.8,108889
4,tt0015324,movie,Sherlock Jr.,Sherlock Jr.,0,1924,<NA>,45,"Action,Comedy,Romance",8.2,58967


<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

#### <font color="#00586D"> Liberación de memoria</font>

En este punto, se pueden eliminar de la memoria los `DataFrame` `df_titles` y `df_ratings`. 

In [11]:
df_titles.memory_usage(deep=True)

Index                   132
tconst            745934134
titleType         731840696
                    ...    
endYear            56087165
runtimeMinutes     56087165
genres            764405392
Length: 10, dtype: int64

In [12]:
import gc

del df_titles
del df_ratings

gc.collect();

In [13]:
df_movies.to_feather("df_movies.feather")  #storing merged df for future loads


In [2]:
import pandas as pd
df_movies = pd.read_feather("df_movies.feather")

In [3]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4976 entries, 0 to 4975
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tconst          4976 non-null   object 
 1   titleType       4976 non-null   string 
 2   primaryTitle    4976 non-null   object 
 3   originalTitle   4976 non-null   string 
 4   isAdult         4976 non-null   Int32  
 5   startYear       4976 non-null   Int32  
 6   endYear         492 non-null    Int32  
 7   runtimeMinutes  4955 non-null   Int32  
 8   genres          4976 non-null   string 
 9   averageRating   4976 non-null   float64
 10  numVotes        4976 non-null   int64  
dtypes: Int32(4), float64(1), int64(1), object(2), string(3)
memory usage: 369.4+ KB


<div align="right">
<a href="#indice"><font size=5 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<a id="section2"></a>
## <font color="#00586D"> 2. Obtención de datos de The Movie Database (TMDb)</font>
<br>





_The Movie Database ([TMDB](https://www.themoviedb.org))_ es otro recurso en la web que contiene información sobre películas y series, y se planteó como una alternativa a _IMDb_. A día de hoy, su uso como medio de recogida de críticas y valoraciones _es muy limitado_. Sin embargo,  contiene información completa y actualizada sobre películas, y proporciona una API REST muy completa y bien documentada. En este proyecto utilizaremos la API de _TMDb_ para ampliar el conjunto inicial de datos. 

La API de _TMDb_ requiere auntentificación, por lo que para trabajar con ella es necesario, en primer lugar, disponer de un usuario. Una vez hecho el registro en el sitio, es necesario solicitar una clave para el uso de la API. Las instrucciones detalladas se muestran en esta página ([enlace](https://developers.themoviedb.org/3/getting-started/introduction)). Este proceso es sencillo, y básicamente consiste en 3 pasos:

1. Entrar en la configuración de la cuenta personal.
2. Entrar en el menú de la API.
3. Crear la API. 

Posteriormente es posible acceder, dentro de la configuración de la cuenta, a la sección correspondiente a la API. En ella se encuentran varios tokens de acceso. En este proyecto necesitamos el primero de ellos, etiquetado como _Clave de la API (v3 auth)_. Copiar este token y asignarlo a la variable `token`.


In [54]:
import os
token = os.environ.get("TMDB_TOKEN", "<YOUR_TMDB_TOKEN>")  # TMDb v3 API key

La API de _TMDb_ es accesible a través de la dirección base `https://api.themoviedb.org/3/`. En la documentación ([enlace](https://developers.themoviedb.org/3/getting-started/introduction)) pueden consultarse todos los _endpoints_ que proporciona. Además, desde este sitio es posible probar los métodos, y generar las llamadas en diversos lenguajes. En el caso de _Python_ se generan dos alternativas. La primera utiliza la librería `http.client`, mientras que la segunda utiliza `requests`.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> Recientemente se ha añadido una API posterior que básicamente modifica el modo de acceso, pero que **no** utilizaremos porque, además, no es estable. 
</div>


In [31]:
df_movies.set_index('tconst',inplace=True)

KeyError: "None of ['tconst'] are in the columns"

In [32]:
df_movies.head()

,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
tconst,,,,,,,,,,
tt0000417,short,A Trip to the Moon,Le voyage dans la lune,0,1902,<NA>,13,"Action,Adventure,Comedy",8.1,57200
tt0010323,movie,The Cabinet of Dr. Caligari,Das Cabinet des Dr. Caligari,0,1920,<NA>,67,"Horror,Mystery,Thriller",8.0,71863
tt0012349,movie,The Kid,The Kid,0,1921,<NA>,68,"Comedy,Drama,Family",8.2,137619
tt0013442,movie,Nosferatu: A Symphony of Horror,"Nosferatu, eine Symphonie des Grauens",0,1922,<NA>,94,"Fantasy,Horror",7.8,108889
tt0015324,movie,Sherlock Jr.,Sherlock Jr.,0,1924,<NA>,45,"Action,Comedy,Romance",8.2,58967


Para ilustrar algunos ejemplos en el proyecto, se utilizará la película con identificador`'tt0068646'`: ___The Godfather___.

In [6]:
tgf_movie_id = 'tt0068646'
df_movies.loc[tgf_movie_id]

titleType                 movie
primaryTitle      The Godfather
originalTitle     The Godfather
isAdult                       0
startYear                  1972
endYear                    <NA>
runtimeMinutes              175
genres              Crime,Drama
averageRating               9.2
numVotes                2063067
Name: tt0068646, dtype: object

In [6]:
import requests
url = "https://api.themoviedb.org/3/movie/tt0068646?external_source=imdb_id&language=en_US"
response = requests.get(url, headers=headers)
tgf_data = response.json()

# Esta es una de las formas de mostrar un json en formato "amigable"
print(json.dumps(tgf_data, indent=3))

{
   "adult": false,
   "backdrop_path": "/tmU7GeKVybMWFButWEGl2M4GeiP.jpg",
   "belongs_to_collection": {
      "id": 230,
      "name": "The Godfather Collection",
      "poster_path": "/zqV8MGXfpLZiFVObLxpAI7wWonJ.jpg",
      "backdrop_path": "/mDMCET9Ens5ANvZAWRpluBsMAtS.jpg"
   },
   "budget": 6000000,
   "genres": [
      {
         "id": 18,
         "name": "Drama"
      },
      {
         "id": 80,
         "name": "Crime"
      }
   ],
   "homepage": "http://www.thegodfather.com/",
   "id": 238,
   "imdb_id": "tt0068646",
   "origin_country": [
      "US"
   ],
   "original_language": "en",
   "original_title": "The Godfather",
   "overview": "Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.",
   "popularity": 165.902,
 

<div align="right">
<a href="#indice"><font size=4 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>



<a id="section21"></a>
### <font color="#00586D"> 2.1 Obtención de información sobre las películas </font>
<br>

La llamada `GET /movie/{movie_id}` permite obtener información sobre una película. El campo `movie_id` corresponde al identificador de la película, que puede ser el propio de _TMDb_, o el identificador en _IMDb_ (que es del que disponemos en este punto). Uno de los parámetros que toma  es `language`, que en este proyecto se fijará a `en_US`. 

In [3]:
import os
import requests
import json

url = "https://api.themoviedb.org/3/movie/tt0068646?external_source=imdb_id&language=en_US"

headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {os.environ.get('TMDB_V4_BEARER', '<YOUR_TMDB_V4_BEARER>')}"
}

In [17]:
tgf_data = 


'Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.'

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener y mostrar la información relativa la película _The Godfather_  (cuyo código en _IMDb_ es `'tt0068646'` y está almacenado en `tgf_movie_id`).

In [135]:

response = requests.get(url, headers=headers)
tgf_data = response.json()

# Esta es una de las formas de mostrar un json en formato "amigable"
print(json.dumps(tgf_data, indent=3))

{
   "adult": false,
   "backdrop_path": "/tmU7GeKVybMWFButWEGl2M4GeiP.jpg",
   "belongs_to_collection": {
      "id": 230,
      "name": "The Godfather Collection",
      "poster_path": "/zqV8MGXfpLZiFVObLxpAI7wWonJ.jpg",
      "backdrop_path": "/mDMCET9Ens5ANvZAWRpluBsMAtS.jpg"
   },
   "budget": 6000000,
   "genres": [
      {
         "id": 18,
         "name": "Drama"
      },
      {
         "id": 80,
         "name": "Crime"
      }
   ],
   "homepage": "http://www.thegodfather.com/",
   "id": 238,
   "imdb_id": "tt0068646",
   "origin_country": [
      "US"
   ],
   "original_language": "en",
   "original_title": "The Godfather",
   "overview": "Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.",
   "popularity": 179.703,
 

In [138]:
tgf_data.keys()

dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count'])

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Puede observarse que la película se caracteriza por dos idenfificadores, `id`, que es el propio de TMDb, e `imdb_id`, que es el identificador en _IMDb_. Esto puede dar lugar a errores, por lo que más adelante se hará los cambios correspondientes para utilizar solamente el último. 
</div>


 <font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Mostrar el resumen de la película, que se encuentra en el campo `overview`.

In [139]:
print(tgf_data['overview'])

Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> Por si tenéis curiosidad o aficción, el campo `poster_path` del objeto JSON contiene el identificador del poster de la película, que puede ser accedido como `http://image.tmdb.org/t/p/w185/poster_path`. Existen varios tipos y tamaños de posters.  
</div>


In [140]:
Image = 'http://image.tmdb.org/t/p/w185'+tgf_data['poster_path']
Image

'http://image.tmdb.org/t/p/w185/3bhkrj58Vtu7enYsRolD1fZdja1.jpg'

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Una vez implementado el acceso a los datos de una película mediante la API de _TMDb_, se obtendrán los datos relativos a todas las películas referenciadas en `df_movies`, y se almacenarán en un diccionario que se denominará `movies_data`, en el que la clave es el identificador de la película, y el valor corresponde a la información descargada.

<div class="alert alert-block alert-danger">
    
<i class="fa fa-exclamation-triangle" aria-hidden="true"></i> Cada llamada a la API requiere aproximadamente 0.5 segundos. Por tanto, la obtención de los datos para todas las películas puede requerir, si se hace secuencialmente, entorno a 2000 segundos. Debido a esto, y de cara a facilitar el desarrollo del trabajo en distintas sesiones, y evitar repetir las llamadas, es conveniente almacenar los resultados en un archivo, al que se denominará `data/backup/movie_data.json`. 
<br>
    
    
Por otra parte, los errores también devuelven información en un *JSON*. Hay controlar que no se incluya en `movies_data` la información correspondiente a películas cuyos datos no se hayan podido recuperar, esto se puede hacer comprobando que (`response.status_code==200`).
</div>

In [27]:
import os
x = 0
# Si no se han descargado todavía los datos, los descarga y almacena en el archivo.
if not os.path.isfile('data/backup/movie_data.json'):
    # Descarga los datos uno por uno
    movie_data = { }
    for movie_id in df_movies.index:
        url = f"https://api.themoviedb.org/3/movie/{movie_id}?external_source=imdb_id&language=en_US"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            movies_data[movie_id] = response.json()
            print(x)
            x += 1
            continue
            
       
        
    # Almacena movie_data en el archivo json.
    with open('data/backup/movie_data.json',"w") as movie_data_file:
         json.dump(movies_data, movie_data_file, indent=4, ensure_ascii=False) 
        
# Si se habían descargado anteriormente, y el archivo está disponible, los lee. 
else:
    with open('data/backup/movie_data.json','r') as movie_data_file:
        movie_data = json.load(movie_data_file) 
    
    
print(f"El de títulos obtenidos es {len(movie_data)}")

El de títulos obtenidos es 4295


Muestra el título y fecha de la película ejemplo. Ahora a partir del diccionario creado.

In [71]:
tgf_data = movies_data['tt0068646']
tgf_data['movie_results'][0]['title'], tgf_data['movie_results'][0]['release_date'], 


('The Godfather', '1972-03-14')

<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<div align="right">
<a href="#indice"><font size=4 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>


<a id="section22"></a>
### <font color="#00586D"> 2.2 Almacenamiento en MongoDB</font>
<br>

Existen bases de datos _NoSQL_ de distinta naturaleza. En _MongoDB_, los datos se organizan en colecciones de documentos cuyo formato es similar al JSON. Con el fin de tener un almacén de información, se va a utilizar esta herramienta, vista en el módulo. Se almacenarán los datos obtenidos anteriormente, y nuevos datos que obtendremos después.

Aunque es posible instalar _MongoDB_ en local, existe un servicio gratuito en la nube, denominado _MongoDB Atlas_ [(enlace)](https://www.mongodb.com/cloud/atlas) en el que se puede desplegar una base de datos. Por otra parte, la librería `pymongo` permite interactuar con bases de datos _MongoDB_. En este tutorial se explica como crear una base de datos en MongoDB Atlas y acceder a ella mediante `pymongo` [(enlace)](https://www.kaggle.com/ashishsaxena2209/tutorial-to-use-mongodb-atlas-cloud-service).

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Crear una base de datos _MongoDB_ denominada `dbmovies` y una colección denominada `movies` (**podéis hacerlo directamente desde la web**). Acceder a la base de datos y almacenar la referencia en la variable `dbmovies`.

<div class="alert alert-block alert-danger">
    
<i class="fa fa-exclamation-triangle" aria-hidden="true"></i> Puede aparecer un error de conexión. Una forma de solucionarlo es importar el paquete `certifi` y añadir el argumento `tlsCAFile=certifi.where()` en la creación y conexión del cliente.
</div>

In [4]:
!pip install "pymongo[srv]"

   ---------------------------------------- 0.0/882.3 kB ? eta -:--:--
   --------------------------------------- 882.3/882.3 kB 13.2 MB/s eta 0:00:00


In [5]:
import certifi
from pymongo import MongoClient



In [6]:
import os
import urllib.parse
password = urllib.parse.quote_plus(os.environ.get("MONGODB_PASSWORD", "<YOUR_MONGODB_PASSWORD>"))  # This encodes the special character (@)
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

uri = f"mongodb+srv://{os.environ.get('MONGODB_USER', '<YOUR_USER>')}:{password}@{os.environ.get('MONGODB_CLUSTER', '<YOUR_CLUSTER>.mongodb.net')}/?retryWrites=true&w=majority"

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Insertar los datos de `movies_data` en esta colección denominada `dbmovies.movies` mediante el método `dbmovies.movies.insert_many()`. Debe tenerse en cuenta para ello que **una colección es una secuencia de documentos**, no un diccionario, por lo que no tenéis que añadir la estructura tal cual, sino una colección con los valores del diccionario. En este caso, perder la clave no supone un problema porque el propio documento asociado a cada película contiene el campo `imdb_id`.


Es necesario inspeccionar la colección en _MongoDB Atlas_ para comprobar que la operación es correcta. Otra opción es contar los documentos para ver que coincide el tamaño de la colección con el de `movie_data`. En caso de que no coincida, puede utilizarse `.drop()` para eliminar la colección, y volverla a crear posteriormente.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> En realidad en este caso, como el conjunto de datos es pequeño, podéis simplificar y borrar en cualquier caso la colección entera con `.drop()` antes de volverla a escribir. Es importante poner `;` al final de la última sentencia para evitar que se imprima la salida.
</div>

<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

In [7]:
dbmovies = client["dbmovies"]  

In [36]:
dbmovies = client["dbmovies"]  
collection = dbmovies["movies"]  
collection.drop()

with open("data/backup/movie_data.json", "r", encoding="utf-8") as file:
    movie_data = json.load(file)  

movies_list = list(movie_data.values())

collection.drop()

insert_result = collection.insert_many(movies_list)

print(f"Inserted {len(insert_result.inserted_ids)} documents into MongoDB")

Inserted 4295 documents into MongoDB


<div align="right">
<a href="#indice"><font size=4 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>



<a id="section23"></a>
### <font color="#00586D"> 2.3 Descarga de los créditos</font>
<br>

Los créditos de una película se refieren a la información sobre todas las personas que participan en ella, desde actores, al director, guionista, etc. La llamada `GET /movie/{movie_id}/credits`, de la API de _TMDb_, devuelve los créditos de la película con identificador `movie_id`.  

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener y mostrar los créditos para la película _The Godfather_. 

In [13]:
url = f"https://api.themoviedb.org/3/movie/tt0068646/credits"
response = requests.get(url, headers=headers)
tgf_data = response.json()
print(json.dumps(tgf_data,indent=3))

{
   "id": 238,
   "cast": [
      {
         "adult": false,
         "gender": 2,
         "id": 3084,
         "known_for_department": "Acting",
         "name": "Marlon Brando",
         "original_name": "Marlon Brando",
         "popularity": 27.275,
         "profile_path": "/fuTEPMsBtV1zE98ujPONbKiYDc2.jpg",
         "cast_id": 146,
         "character": "Don Vito Corleone",
         "credit_id": "6489aa85e2726001072483a9",
         "order": 0
      },
      {
         "adult": false,
         "gender": 2,
         "id": 1158,
         "known_for_department": "Acting",
         "name": "Al Pacino",
         "original_name": "Al Pacino",
         "popularity": 50.623,
         "profile_path": "/fMDFeVf0pjopTJbyRSLFwNDm8Wr.jpg",
         "cast_id": 147,
         "character": "Michael Corleone",
         "credit_id": "6489aa936f8d9500afdf219c",
         "order": 1
      },
      {
         "adult": false,
         "gender": 2,
         "id": 3085,
         "known_for_department": "

Para interpretar la estructura JSON, es necesario explorarla progresivamente. En su nivel principal, es un diccionario. Pueden consultarse sus entradas. 

In [14]:
tgf_data.keys()

dict_keys(['id', 'cast', 'crew'])

Puede observarse que la estructura JSON devuelta contiene, para cada película, tres campos principales:

* `id`, con el identificador de la película en `TMDb`.
* `crew`, con el equipo técnico.
* `cast`, con el reparto. 

Se procesarán `crew` y `cast` por separado.

#### <font color="#00586D">Dirección (crew) </font>

La subestructura correspondiente al campo `crew` es una lista en la que se ordenan todos los componentes de la dirección y equipo técnico. Cada uno de ellos se representa, a su vez, mediante un diccionario. La siguiente celda de código muestra las tres primeras entradas.

In [15]:
tgf_data['crew'][0:3]

[{'adult': False,
  'gender': 2,
  'id': 457,
  'known_for_department': 'Production',
  'name': 'Albert S. Ruddy',
  'original_name': 'Albert S. Ruddy',
  'popularity': 3.828,
  'profile_path': '/jSkLkR79OmqYHg0kx4WWXI1TYPP.jpg',
  'credit_id': '52fe422bc3a36847f80093cf',
  'department': 'Production',
  'job': 'Producer'},
 {'adult': False,
  'gender': 2,
  'id': 3083,
  'known_for_department': 'Writing',
  'name': 'Mario Puzo',
  'original_name': 'Mario Puzo',
  'popularity': 6.928,
  'profile_path': '/lEsT1uCZAZg1nYDQe3Fsj9CalzT.jpg',
  'credit_id': '52fe422bc3a36847f80093d5',
  'department': 'Writing',
  'job': 'Screenplay'},
 {'adult': False,
  'gender': 2,
  'id': 1776,
  'known_for_department': 'Directing',
  'name': 'Francis Ford Coppola',
  'original_name': 'Francis Ford Coppola',
  'popularity': 18.88,
  'profile_path': '/IwGgkmW6IoJ9vuNF0T9CU3FYUX.jpg',
  'credit_id': '52fe422bc3a36847f80093db',
  'department': 'Writing',
  'job': 'Screenplay'}]

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener, a partir de la estructura `tgf_data` obtenida anteriormente (los créditos de _The GodFather_), la lista de entradas en las que el campo `job=='Director'` (en este caso, una lista de tamaño uno, pero puede haber varios en otras películas). Utilizar para ello la función `filter`.

In [16]:
director = list(filter(lambda p:p['job'] == 'Director',tgf_data['crew'] ))
print(director)

[{'adult': False, 'gender': 2, 'id': 1776, 'known_for_department': 'Directing', 'name': 'Francis Ford Coppola', 'original_name': 'Francis Ford Coppola', 'popularity': 18.88, 'profile_path': '/IwGgkmW6IoJ9vuNF0T9CU3FYUX.jpg', 'credit_id': '5e92505cccb15f00136de455', 'department': 'Directing', 'job': 'Director'}]


#### <font color="#00586D">Reparto (cast) </font>

El tercer campo de la estructura devuelta por la llamada `GET /movie/{movie_id}/credits` se denomina `cast` y contiene, en una lista, el elenco de actores que han participado en la película cuyo código es `movie_id`.  La siguiente celda muestra las tres primeras entradas.

In [18]:
tgf_data['cast'][0:3]

[{'adult': False,
  'gender': 2,
  'id': 3084,
  'known_for_department': 'Acting',
  'name': 'Marlon Brando',
  'original_name': 'Marlon Brando',
  'popularity': 27.275,
  'profile_path': '/fuTEPMsBtV1zE98ujPONbKiYDc2.jpg',
  'cast_id': 146,
  'character': 'Don Vito Corleone',
  'credit_id': '6489aa85e2726001072483a9',
  'order': 0},
 {'adult': False,
  'gender': 2,
  'id': 1158,
  'known_for_department': 'Acting',
  'name': 'Al Pacino',
  'original_name': 'Al Pacino',
  'popularity': 50.623,
  'profile_path': '/fMDFeVf0pjopTJbyRSLFwNDm8Wr.jpg',
  'cast_id': 147,
  'character': 'Michael Corleone',
  'credit_id': '6489aa936f8d9500afdf219c',
  'order': 1},
 {'adult': False,
  'gender': 2,
  'id': 3085,
  'known_for_department': 'Acting',
  'name': 'James Caan',
  'original_name': 'James Caan',
  'popularity': 32.077,
  'profile_path': '/v3flJtQEyczxENi29yJyvnN6LVt.jpg',
  'cast_id': 148,
  'character': 'Sonny Corleone',
  'credit_id': '6489aabc99259c00ff111136',
  'order': 2}]

Uno de los campos, `order`, representa el orden de importancia en el reparto. Aunque parece que coincide con el orden de las entradas, podría no ser así, por lo que se puede utilizar también para hacer la selección mediante `filter`. 


<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener los tres actores/actrices más importantes en el reparto de la película anterior (el orden del primero es cero). 

In [19]:
top_cast = list(filter(lambda p:p['order']<3,tgf_data['cast']))
top_cast

[{'adult': False,
  'gender': 2,
  'id': 3084,
  'known_for_department': 'Acting',
  'name': 'Marlon Brando',
  'original_name': 'Marlon Brando',
  'popularity': 27.275,
  'profile_path': '/fuTEPMsBtV1zE98ujPONbKiYDc2.jpg',
  'cast_id': 146,
  'character': 'Don Vito Corleone',
  'credit_id': '6489aa85e2726001072483a9',
  'order': 0},
 {'adult': False,
  'gender': 2,
  'id': 1158,
  'known_for_department': 'Acting',
  'name': 'Al Pacino',
  'original_name': 'Al Pacino',
  'popularity': 50.623,
  'profile_path': '/fMDFeVf0pjopTJbyRSLFwNDm8Wr.jpg',
  'cast_id': 147,
  'character': 'Michael Corleone',
  'credit_id': '6489aa936f8d9500afdf219c',
  'order': 1},
 {'adult': False,
  'gender': 2,
  'id': 3085,
  'known_for_department': 'Acting',
  'name': 'James Caan',
  'original_name': 'James Caan',
  'popularity': 32.077,
  'profile_path': '/v3flJtQEyczxENi29yJyvnN6LVt.jpg',
  'cast_id': 148,
  'character': 'Sonny Corleone',
  'credit_id': '6489aabc99259c00ff111136',
  'order': 2}]

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Puede observarse que cada persona tiene asociado un `id`en _TMDb_. Por este motivo, y por simplificar, se utilizará como identificador `imdb_id`.</div>

#### <font color="#00586D">Descarga </font>

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Descargar los datos con los créditos para cada una de las películas en `movie_data`. Las descargas pueden hacer de forma distinta, almacenando cada respuesta directamente en una colección denominada `credits` en `dbmovies`. Además, por simplificar, se sustituirá el campo `id`, que contiene el identificador en _TMDb_, por un campo denominado `imdb_id`, que contiene el identificador en _IMDb_. Esto implica hacer un cambio en el objeto leído desde la API antes de almacenarlo.



<div class="alert alert-block alert-danger">
    
<i class="fa fa-exclamation-triangle" aria-hidden="true"></i> 
Antes de hacer cada petición se ha de comprobar que los créditos no esten almacenados (`{'imdb_id':movie_id}`). Solamente se ha de almacenar la información si la respuesta es correcta (`response.status_code==200`).
</div>

In [24]:
dbmovies.list_collections()

In [28]:
movie_data['tt0000417']

{'adult': False,
 'backdrop_path': '/g67r1eiQD3ERSEQFCFkSn7TeGw5.jpg',
 'belongs_to_collection': None,
 'budget': 5985,
 'genres': [{'id': 12, 'name': 'Adventure'},
  {'id': 878, 'name': 'Science Fiction'}],
 'homepage': '',
 'id': 775,
 'imdb_id': 'tt0000417',
 'origin_country': ['FR'],
 'original_language': 'fr',
 'original_title': 'Le Voyage dans la Lune',
 'overview': 'Professor Barbenfouillis and five of his colleagues from the Academy of Astronomy travel to the Moon aboard a rocket propelled by a giant cannon. Once on the lunar surface, the bold explorers face the many perils hidden in the caves of the mysterious planet.',
 'popularity': 16.679,
 'poster_path': '/9o0v5LLFk51nyTBHZSre6OB37n2.jpg',
 'production_companies': [{'id': 7159,
   'logo_path': None,
   'name': 'Star Film',
   'origin_country': 'FR'}],
 'production_countries': [{'iso_3166_1': 'FR', 'name': 'France'}],
 'release_date': '1902-06-21',
 'revenue': 0,
 'runtime': 15,
 'spoken_languages': [{'english_name': 'No La

In [38]:
movies_id = [movie_data['imdb_id'] for movie_data in movie_data.values()]
collection_credits = dbmovies['credits']

# query a la coleccion y genera una lista con los imdb_id
existing_ids = [doc["imdb_id"] for doc in collection_credits.find({}, {"imdb_id": 1, "_id": 0})]

#itera sobre los elementos de movies_id
for movie_id in movies_id:
    # comprueba que la movie_id to este ya en la coleccion
    if movie_id not in existing_ids:
        url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            credits_movie = response.json()
            credits_movie["imdb_id"] = movie_id 
            del credits_movie["id"]  
            collection_credits.insert_one(credits_movie)  # Insert into MongoDB


<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

In [4]:
#code to get movie credits
url = f"https://api.themoviedb.org/3/movie/tt0068646/credits"
response = requests.get(url, headers=headers)
tgf_data = response.json()
print(json.dumps(tgf_data,indent=3))

{
   "id": 238,
   "cast": [
      {
         "adult": false,
         "gender": 2,
         "id": 3084,
         "known_for_department": "Acting",
         "name": "Marlon Brando",
         "original_name": "Marlon Brando",
         "popularity": 24.583,
         "profile_path": "/fuTEPMsBtV1zE98ujPONbKiYDc2.jpg",
         "cast_id": 146,
         "character": "Don Vito Corleone",
         "credit_id": "6489aa85e2726001072483a9",
         "order": 0
      },
      {
         "adult": false,
         "gender": 2,
         "id": 1158,
         "known_for_department": "Acting",
         "name": "Al Pacino",
         "original_name": "Al Pacino",
         "popularity": 53.187,
         "profile_path": "/fMDFeVf0pjopTJbyRSLFwNDm8Wr.jpg",
         "cast_id": 147,
         "character": "Michael Corleone",
         "credit_id": "6489aa936f8d9500afdf219c",
         "order": 1
      },
      {
         "adult": false,
         "gender": 2,
         "id": 3085,
         "known_for_department": "

<div align="right">
<a href="#indice"><font size=4 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>



<a id="section24"></a>
### <font color="#00586D"> 2.4 Datos de personas</font>
<br>

La base de datos _MongoDB_ `cidaendb` contiene colecciones relacionadas con los conjuntos de datos que se están manejando en este trabajo. En concreto, la colección `people` proporciona información relativa a las personas más importantes incluidas en los créditos, y contiene datos como nombre, nacimiento, e incluso una pequeña biografía. 

La siguiente _url_ permite hacer una llamada a `cidaendb.people.findOne()` a través de una API que hemos publicado. Entre otros parámetros admite, a través de la estructura `payload`, un diccionario denominado `projection` con los campos que se han de devolver, y otro denominado `filter` con los valores que se han de utilizar para seleccionar. Para acceder a este `endpoint` se proporciona una `api-key`.



<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: La API de _TMDb_ permite acceder esta información mediante la llamada `GET person/{people_id}`, que devuelve una estructura con la información sobre la persona con identificador de la persona en la plataforma, `people_id`. 
    
</div>

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Descargar el nombre (`name`), biografía (`biography`), y `profile_path` del actor con `id` 3084. Almacenar el resultado (campo `document` de la estructura devuelta) en la variable `mb_data`.

In [7]:
import os
url = "https://eu-west-2.aws.data.mongodb-api.com/app/data-lioff/endpoint/data/v1/action/findOne"

payload = json.dumps({
    "collection": "people",
    "database": "cidaendb",
    "dataSource": "Cluster0",
    "projection": {
        "name": 1,
        "biography:":1,
        "profile_path":1
    },
    "filter": {
        "id": 3084
    }    
})
headers = {
  'Content-Type': 'application/json',
  'Access-Control-Request-Headers': '*',
  'api-key': os.environ.get('CIDAEN_API_KEY', '<YOUR_CIDAEN_API_KEY>'),
}

response = requests.request("POST", url, headers=headers, data=payload)

mb_data = response.json()['document']
mb_data

{'_id': '673b67a01520298180dd968a',
 'name': 'Marlon Brando',
 'profile_path': '/fuTEPMsBtV1zE98ujPONbKiYDc2.jpg'}

Uno de los campos, `profile_path` contiene la ruta a una foto de la persona. Esta se construye como `http://image.tmdb.org/t/p/w185/profile_path`. 

In [10]:
Image = ('http://image.tmdb.org/t/p/w185/'+mb_data['profile_path'])
Image

'http://image.tmdb.org/t/p/w185//fuTEPMsBtV1zE98ujPONbKiYDc2.jpg'

Mediante la siguiente _url_ permite hacer una llamada `cidaendb.people.find()`. Al no utilizar parámetros para filtrar o proyectar, descarga toda la información.  Es necesario especificar `limit:10000` para que baje la colección completa, ya que por defecto limita el número de documentos.  

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Descargar los datos y guardarlos (campo `documents`) en `people_data`.

In [10]:
import os
import json
import requests
url = "https://eu-west-2.aws.data.mongodb-api.com/app/data-lioff/endpoint/data/v1/action/find"

payload = json.dumps({
    "collection": "people",
    "database": "cidaendb",
    "dataSource": "Cluster0",
    "limit" :10000,
})

headers = {
  'Content-Type': 'application/json',
  'Access-Control-Request-Headers': '*',
  'api-key': os.environ.get('CIDAEN_API_KEY', '<YOUR_CIDAEN_API_KEY>'),
}

response = requests.request("POST", url, headers=headers, data=payload)
people_data = response.json()['documents']
len(people_data)

6406

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Almacenar la información de `people_data` en la colección denominada `dbmovies.people`. Como el tamaño de esta no es excesivamente grande, puede borrarse primero la colección `collection.drop()` para evitar errores.

In [50]:
collection = dbmovies["people"]
collection.insert_many(people_data)

KeyboardInterrupt: 

<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

In [12]:
collection_people = dbmovies["people"]
collection_credits = dbmovies['credits']

<div align="right">
<a href="#indice"><font size=5 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<a id="section3"></a>
## <font color="#00586D"> 3. Elaboración de un conjunto de datos estructurado</font>
<br>

Llegados a este punto se dispone de la siguiente información:

* El `DataFrame` `df_movies`, con información sobre las películas,
* La colección `dbmovies.movies`, almacenada en _MongoDB Atlas_, con información adicional sobre las películas,
* La colección `dbmovies.credits`, con los créditos de cada película,
* La colección `dbmovies.people`, con información sobre las personas más importantes que aparecen en los créditos.


El objetivo ahora es disponer los datos en formato estructurado, de forma que sea posible el análisis. Existen dos opciones: Utilizar `DataFrame`, o una base de datos relacional. Puesto que la primera puede ser un paso previo para la segunda, se harán las dos.

<div align="right">
<a href="#indice"><font size=4 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

<a id="section31"></a>
### <font color="#00586D"> 3.1 Creación de `DataFrames`</font>
<br>


En primer lugar,  se añadirá alguna información de `dbmovies.movies` a `df_movies`. Para ello, se renombrará previamente el índice del conjunto `df_movies` a `imdb_id`.

In [21]:
df_movies.index.rename('imdb_id', inplace=True)
df_movies

,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
imdb_id,,,,,,,,,,
tt0000417,short,A Trip to the Moon,Le voyage dans la lune,0,1902,<NA>,13,"Action,Adventure,Comedy",8.1,57200
tt0010323,movie,The Cabinet of Dr. Caligari,Das Cabinet des Dr. Caligari,0,1920,<NA>,67,"Horror,Mystery,Thriller",8.0,71863
tt0012349,movie,The Kid,The Kid,0,1921,<NA>,68,"Comedy,Drama,Family",8.2,137619
tt0013442,movie,Nosferatu: A Symphony of Horror,"Nosferatu, eine Symphonie des Grauens",0,1922,<NA>,94,"Fantasy,Horror",7.8,108889
tt0015324,movie,Sherlock Jr.,Sherlock Jr.,0,1924,<NA>,45,"Action,Comedy,Romance",8.2,58967
...,...,...,...,...,...,...,...,...,...,...
tt9844522,movie,Escape Room: Tournament of Champions,Escape Room: Tournament of Champions,0,2021,<NA>,88,"Action,Adventure,Horror",5.7,62208
tt9866072,movie,Holidate,Holidate,0,2020,<NA>,104,"Comedy,Romance",6.1,80693
tt9893250,movie,I Care a Lot,I Care a Lot,0,2020,<NA>,118,"Comedy,Crime,Drama",6.4,150028


Por simplicidad se va a añadir solamente el presupuesto (`budget`) y la recaudación (`revenue`). 

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Descargar esta información a una estructura denominada `movies_data` y añadirla en las columnas correspondientes en `df_movies`. Borrar también las columnas `titleType`, `originalTitle` y `endYear`.

<br>

<details>    
<summary>
    <font size=3 color="#00586D"> <i class="fa fa-info-circle"  aria-hidden="true"></i> </font>
</summary>
   
* Se trata de bajar primero los datos de `dbmovies.movies` y almacenarlos en una estructura denominada `movies_data` (que es un cursor).
* Se debe indicar también que se descargue `imdb_id`, ya que esto permitirá agregar los datos a `df_movies`.
* Por higiene, es conveniente indicar explícitamente que no se descargue `_id`.
* Una vez descargados los datos, puede crearse un `DataFrame` intermedio, indexado por `imdb_id`.
* Por último, hay que fundir este `DataFrame` con `df_movies`, almacenando el resultado en `df_movies`.

</details>   

In [22]:
movies_collection = dbmovies["movies"]
movies_data = movies_collection.find({},{"imdb_id":1, "budget":1,"revenue":1,"_id":0})
movies_data = pd.DataFrame(list(movies_data))
movies_data.set_index("imdb_id",inplace=True)
movies_data.head()

df_movies = df_movies.merge (movies_data, on = "imdb_id",how = "inner") #hago un inner merge para mantener solo los casos en los que existen datos en movies_data
df_movies.head()

,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,budget,revenue
imdb_id,,,,,,,,,,,,
tt0000417,short,A Trip to the Moon,Le voyage dans la lune,0,1902,<NA>,13,"Action,Adventure,Comedy",8.1,57200,5985,0
tt0010323,movie,The Cabinet of Dr. Caligari,Das Cabinet des Dr. Caligari,0,1920,<NA>,67,"Horror,Mystery,Thriller",8.0,71863,18000,8811
tt0012349,movie,The Kid,The Kid,0,1921,<NA>,68,"Comedy,Drama,Family",8.2,137619,250000,2500000
tt0013442,movie,Nosferatu: A Symphony of Horror,"Nosferatu, eine Symphonie des Grauens",0,1922,<NA>,94,"Fantasy,Horror",7.8,108889,0,19054
tt0015324,movie,Sherlock Jr.,Sherlock Jr.,0,1924,<NA>,45,"Action,Comedy,Romance",8.2,58967,0,0


In [26]:
df_movies = df_movies[df_movies["titleType"] == "movie"]
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4265 entries, tt0010323 to tt9893250
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   titleType       4265 non-null   string 
 1   primaryTitle    4265 non-null   object 
 2   originalTitle   4265 non-null   string 
 3   isAdult         4265 non-null   Int32  
 4   startYear       4265 non-null   Int32  
 5   endYear         0 non-null      Int32  
 6   runtimeMinutes  4265 non-null   Int32  
 7   genres          4265 non-null   string 
 8   averageRating   4265 non-null   float64
 9   numVotes        4265 non-null   int64  
 10  budget          4265 non-null   int64  
 11  revenue         4265 non-null   int64  
dtypes: Int32(4), float64(1), int64(3), object(1), string(3)
memory usage: 383.2+ KB


In [27]:
df_movies = df_movies.drop(columns=['titleType', 'originalTitle', 'endYear'])

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> La colección `dbmovies.people` contiene información sobre las personas. Descargar los datos correspondientes a `id`, `name`, `gender`, `popularity`, y `place_of_birth`, y almacenar el cursor en `people_data` (indicar también de forma explícita que no se descargue `_id`).

In [28]:
collection_people = dbmovies["people"]
projection = {"id":1,"name":1,"gender":1,"populariy":1,"place_of_birth":1,"_id":0}
people_data = list(collection_people.find({},projection))
people_data[0:5]

[{'gender': 2,
  'id': 13848,
  'name': 'Charlie Chaplin',
  'place_of_birth': 'Walworth, London, England, UK'},
 {'gender': 2,
  'id': 9076,
  'name': 'F. W. Murnau',
  'place_of_birth': 'Bielefeld, North-Rhine-Westphalia, Germany'},
 {'gender': 2,
  'id': 8635,
  'name': 'Buster Keaton',
  'place_of_birth': 'Piqua, Kansas, USA'},
 {'gender': 2,
  'id': 11572,
  'name': 'Carl Theodor Dreyer',
  'place_of_birth': 'Copenhagen, Denmark'},
 {'gender': 2,
  'id': 30008,
  'name': 'Leo McCarey',
  'place_of_birth': 'Los Angeles, California, USA'}]

Con el fin de tener algo más de orden y claridad en los datos, se sustituye el campo que originalmente se denomina `id` por `people_id`.

In [29]:
for person_data in people_data:
    person_data['people_id'] = person_data.pop('id')

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Crear un `DataFrame` denominado `df_people` a partir de `people_data`, utilizando el campo `people_id` como índice.

In [30]:
df_people = pd.DataFrame(people_data).set_index("people_id")
df_people

,gender,name,place_of_birth
people_id,,,
13848,2,Charlie Chaplin,"Walworth, London, England, UK"
9076,2,F. W. Murnau,"Bielefeld, North-Rhine-Westphalia, Germany"
8635,2,Buster Keaton,"Piqua, Kansas, USA"
11572,2,Carl Theodor Dreyer,"Copenhagen, Denmark"
30008,2,Leo McCarey,"Los Angeles, California, USA"
...,...,...,...
35027,3,Ryan Simpkins,"New York City, New York, USA"
1377686,1,Sarah Goldberg,"Vancouver, British Columbia, Canada"
6198,2,Vondie Curtis-Hall,"Detroit, Michigan, USA"


Por último, `dbmovies.credits` contiene información relativa a los créditos. Un vistazo en _MongoDB Atlas_ permite apreciar que el tamaño de esa colección supera los 150 MB. En realidad, mucha información es irrelevante, por lo que se limitará la información a descargar a:

* El campo `imdb_id`
* El director (campo `job` de los elementos en el campo `crew`)
* Los 3 actores o actrices principales (el `order` de los elementos en el campo `cast` puede tomar valores 0, 1, y 2).

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Descargar esta información y guardarla en una lista denominada `credits_data`.

<br>

<details>    
<summary>
    <font size=3 color="#00586D"> <i class="fa fa-info-circle"  aria-hidden="true"></i> </font>
</summary>
   
* Esta consulta es difícil, ya que hay añadir filtros dentro de los campos especificados en la proyección. Si tenéis dudas podéis utilizar chatGPT (recomendado porque hay que aprender a preguntar)... ¡o preguntarnos a nosotros!

</details>  

In [31]:
collection_credits= dbmovies["credits"]


<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

cedric

In [32]:
pipeline = [
    {
        '$project': {
            '_id': 0,
            'imdb_id': 1,
            'cast': {
                '$filter': {
                    'input': '$cast',
                    'as': 'cast_member',
                    'cond': {'$lte': ['$$cast_member.order', 2]}
                }
            },
            'crew': {
                '$filter': {
                    'input': '$crew',
                    'as': 'crew_member',
                    'cond': {'$eq': ['$$crew_member.job', 'Director']}
                }
            }
        }
    },
    {
        '$project': {
            'imdb_id': 1,
            'cast': {
                '$map': {
                    'input': '$cast',
                    'as': 'cast_member',
                    'in': {
                        'id': '$$cast_member.id',
                        'name': '$$cast_member.name'
                    }
                }
            },
            'crew': {
                '$map': {
                    'input': '$crew',
                    'as': 'crew_member',
                    'in': {
                        'id': '$$crew_member.id',
                        'name': '$$crew_member.name'
                    }
                }
            }
        }
    }
]

credits_data = list(dbmovies.credits.aggregate(pipeline))


In [33]:
pd.DataFrame(credits_data).head().set_index('imdb_id')

,cast,crew
imdb_id,,
tt0000417,"[{'id': 11523, 'name': 'Georges Méliès'}, {'id...","[{'id': 11523, 'name': 'Georges Méliès'}]"
tt0010323,"[{'id': 3000, 'name': 'Werner Krauss'}, {'id':...","[{'id': 2991, 'name': 'Robert Wiene'}]"
tt0012349,"[{'id': 13848, 'name': 'Charlie Chaplin'}, {'i...","[{'id': 13848, 'name': 'Charlie Chaplin'}]"
tt0013442,"[{'id': 9839, 'name': 'Max Schreck'}, {'id': 9...","[{'id': 9076, 'name': 'F. W. Murnau'}]"
tt0015324,"[{'id': 8635, 'name': 'Buster Keaton'}, {'id':...","[{'id': 8635, 'name': 'Buster Keaton'}]"


Una vez descargados los datos, se ha de crear un `DataFrame` denominado `df_credits` que contendrá la información relativa a las personas incluidas en los créditos de cada película, así como su rol. En concreto, este nuevo conjunto de datos constará de tres columnas:

* `imdb_id`: identificador de la película en _IMDb_.
* `people_id`: identificador de la persona en _TMDb_.
* `rol`: de la persona en la película (`cast`/`director`).


Como paso previo a la creación de `df_credits`, pueden obtenerse los identificadores correspondientes a partir de `credits_data`. Es más sencillo hacerlo por separado.

In [34]:
aux_cast = [(credit_mov['imdb_id'], [cast_mem['id'] for cast_mem in credit_mov['cast']]) for credit_mov in credits_data]
aux_crew = [(credit_mov['imdb_id'], [crew_mem['id'] for crew_mem in credit_mov['crew']]) for credit_mov in credits_data]

aux_cast[:3]

#[('tt0010323', [3000, 3001, 590591]),
# ('tt0012349', [13848, 19426, 21301]),
# ('tt0013442', [9839, 9840, 9841])]

[('tt0000417', [11523, 11645, 1271225]),
 ('tt0010323', [3000, 3001, 590591]),
 ('tt0012349', [13848, 19426, 63378])]

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Crear un `DataFrame` denominado `df_cast` a partir de los datos de `df_cast`.

<br>

<details>    
<summary>
    <font size=3 color="#00586D"> <i class="fa fa-info-circle"  aria-hidden="true"></i> </font>
</summary>
   
* Se debe crear un `DataFrame` con dos columnas. La segunda, cuyo índice es `1`, contiene una lista de 3 elementos. Mediante el método `Series.explode(1)` la lista se puede convertir a 3 filas con el mismo valor de la columna `0`.

</details> 

In [35]:
df_cast = pd.DataFrame(aux_cast,columns=['imdb_id','cast']).set_index('imdb_id')
df_cast

,cast
imdb_id,
tt0000417,"[11523, 11645, 1271225]"
tt0010323,"[3000, 3001, 590591]"
tt0012349,"[13848, 19426, 63378]"
tt0013442,"[9839, 9840, 9841]"
tt0015324,"[8635, 14920, 10530]"
...,...
tt9783600,"[74568, 996701, 59017]"
tt9784798,"[206919, 1200864, 88124]"
tt9844522,"[1353827, 116088, 1789788]"


<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Crear un `DataFrame` denominado `df_crew` a partir de `aux_crew`.

In [36]:
df_crew = pd.DataFrame(aux_crew,columns=['imdb_id','crew']).set_index('imdb_id')
df_crew

,crew
imdb_id,
tt0000417,[11523]
tt0010323,[2991]
tt0012349,[13848]
tt0013442,[9076]
tt0015324,[8635]
...,...
tt9783600,[86270]
tt9784798,[1140231]
tt9844522,[1306608]


<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Concatenar `df_cast` y `df_crew` en un nuevo `DataFrame` denominado `df_credits`. Añadir una columna denominada `rol` que tome el valor `cast` o `director` según el origen de las filas. 


<br>

<details>    
<summary>
    <font size=3 color="#00586D"> <i class="fa fa-info-circle"  aria-hidden="true"></i> </font>
</summary>
   
* Este ejercicio requiere que se utilicen adecuadamente los parámetros `keys` y `names` de `pd.concat()`.

</details> 

In [37]:
# 1. Concatenar los dataframes usando keys
df_credits = pd.concat([df_cast, df_crew], 
                      keys=['cast', 'director'],
                      names=['rol', 'imdb_id'])

# 2. Resetear el índice para convertir 'rol' en una columna
df_credits = df_credits.reset_index().set_index('imdb_id')

df_credits

,rol,cast,crew
imdb_id,,,
tt0000417,cast,"[11523, 11645, 1271225]",NaN
tt0010323,cast,"[3000, 3001, 590591]",NaN
tt0012349,cast,"[13848, 19426, 63378]",NaN
tt0013442,cast,"[9839, 9840, 9841]",NaN
tt0015324,cast,"[8635, 14920, 10530]",NaN
...,...,...,...
tt9783600,director,NaN,[86270]
tt9784798,director,NaN,[1140231]
tt9844522,director,NaN,[1306608]


In [38]:
df_people.head()

,gender,name,place_of_birth
people_id,,,
13848,2,Charlie Chaplin,"Walworth, London, England, UK"
9076,2,F. W. Murnau,"Bielefeld, North-Rhine-Westphalia, Germany"
8635,2,Buster Keaton,"Piqua, Kansas, USA"
11572,2,Carl Theodor Dreyer,"Copenhagen, Denmark"
30008,2,Leo McCarey,"Los Angeles, California, USA"


In [45]:
df_people_data = pd.DataFrame(people_data)
df_people_data.set_index('people_id',inplace=True)
df_people_data.head()

,gender,name,place_of_birth
people_id,,,
13848,2,Charlie Chaplin,"Walworth, London, England, UK"
9076,2,F. W. Murnau,"Bielefeld, North-Rhine-Westphalia, Germany"
8635,2,Buster Keaton,"Piqua, Kansas, USA"
11572,2,Carl Theodor Dreyer,"Copenhagen, Denmark"
30008,2,Leo McCarey,"Los Angeles, California, USA"


Por último, existen créditos que hacen referencia a personas cuyos datos no se han podido descargar. Se eliminan para garantizar la consistencia de los datos.

In [110]:
valid_people_ids = set(df_people['people_id'])  
df_credits = df_credits[df_credits['people_id'].isin(valid_people_ids)]





No se si he hecho algo mal, pero no se han removido ningun id por no estar presentes en 'valid_people_ids'

<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

In [111]:
df_credits.head()


,imdb_id,rol,people_id
0,tt0010323,cast,3000
1,tt0010323,cast,3001
2,tt0010323,cast,590591
3,tt0012349,cast,13848
4,tt0012349,cast,19426


In [ ]:
df_people.head()

,gender,name,place_of_birth
people_id,,,
13848,2,Charlie Chaplin,"Walworth, London, England, UK"
9076,2,F. W. Murnau,"Bielefeld, North-Rhine-Westphalia, Germany"
8635,2,Buster Keaton,"Piqua, Kansas, USA"
11572,2,Carl Theodor Dreyer,"Copenhagen, Denmark"
30008,2,Leo McCarey,"Los Angeles, California, USA"


In [56]:
df_credits.head()

,rol,cast,crew
imdb_id,,,
tt0000417,cast,[],NaN
tt0010323,cast,"[3000, 3001, 590591]",NaN
tt0012349,cast,"[13848, 19426, 63378]",NaN
tt0013442,cast,"[9839, 9840, 9841]",NaN
tt0015324,cast,"[8635, 14920, 10530]",NaN


In [57]:
df_credits = df_credits.explode('cast')
df_credits = df_credits.explode('crew')

df_credits['people_id'] = df_credits['cast'].combine_first(df_credits['crew'])

df_credits = df_credits.drop(columns=['cast', 'crew']).dropna().reset_index()


In [112]:
df_credits.set_index('imdb_id',inplace=True)

In [113]:
df_credits.head()

,rol,people_id
imdb_id,,
tt0010323,cast,3000
tt0010323,cast,3001
tt0010323,cast,590591
tt0012349,cast,13848
tt0012349,cast,19426


<div align="right">
<a href="#indice"><font size=4 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

<a id="section32"></a>
### <font color="#00586D"> 3.2 Creación de una base de datos relacional</font>
<br>


Llegados a este punto, se dispone de los conjuntos de datos con información sobre las películas (`df_movies`), qué personas que participan en cada película (`df_credits`) e información adicional sobre esas personas (`df_people`). A continuación, se almacenará esta información en una base de datos relacional (SQLite). Una de las características de estas bases de datos es que minimizan la redundancia de datos. Para ello, es importante definir correctamente el esquema. En este caso, constará de tres tablas:


* `MOVIES`, que contiene los campos relativos a cada película:
    * `imdb_id`, identificador (clave primaria)
    * `primaryTitle`, título
    * `isAdult`, si es película de adultos
    * `startYear`, año de estreno
    * `runtimeMinutes`, duración
    * `genres`, géneros
    * `averageRating`, valoración media
    * `numVotes`, número de votos
    * `budget`, presupuesto
    * `revenue`, recaudación.

    
* `PEOPLE`, que contiene los datos de interés relativos a cada persona:

    * `people_id`, identificador de cada persona (clave primaria)
    * `gender`, género
    * `name`, nombre
    * `place_of_birth`, lugar de nacimiento
    * `popularity`, popularidad en el momento de la descarga de los datos.
    
    
* `CREDITS` establede una relación (muchos a muchos) entre personas y películas. Contiene los siguientes campos:
    * `imdb_id`, identificador de la película
    * `people_id`, identificador de la persona
    * `rol`, rol de la persona en la película ("_director_" o "_cast_")
    
    
Con el fin de garantizar la integridad de los datos, se definen claves foráneas en los campos `imdb_id` y `people_id` de CREDITS, que apuntan a las claves primarias de las tablas `MOVIES` y `PEOPLE` respectivamente. 

<div class="alert alert-block alert-danger">
    
<i class="fa fa-exclamation-triangle" aria-hidden="true"></i>
Surge un problema a la hora de usar claves foráneas cuando se trabaja con Pandas y SQLite. Por un lado, los métodos de exportación de `DataFrame` a SQL no permiten definir este tipo de claves (ni primarias). Por otra parte, SQLite (no otras como MySQL) no acepta `ALTER TABLE` para hacer cambios a posteriori. Debido a esto se trabajará en dos pasos:
    
1. Se define la estructura de las tablas mediante SQL
2. Se volcarán los datos desde Pandas a esas tablas existentes.
</div>

In [62]:
from sqlalchemy import create_engine, text
import os

# Por agilidad, se borra la base de datos, aunque se podría definir una condición
# en las siguientes celdas.
try:
    os.remove('./data/movies.db')
except:
    pass

# Crea una conexión a una base de datos sqlite
engine = create_engine('sqlite:///data/movies.db')  

In [71]:
with engine.connect() as connection:
    connection.execute(text("PRAGMA foreign_keys = ON"))

In [69]:
with engine.connect() as connection:
    connection.execute(text("PRAGMA foreign_keys = ON"))
    

    query = text(""" CREATE TABLE IF NOT EXISTS MOVIES(
                        imdb_id TEXT PRIMARY KEY, 
                        primaryTitle TEXT, 
                        isAdult INTEGER, 
                        startYear INTEGER, 
                        runtimeMinutes INTEGER, 
                        genres TEXT, 
                        averageRating FLOAT, 
                        numVotes BIGINT, 
                        budget BIGINT, 
                        revenue BIGINT) 
                        """ )
    

    connection.execute(query)
    
    
    query = text(""" CREATE TABLE IF NOT EXISTS PEOPLE (
                        people_id BIGINT PRIMARY KEY, 
                        gender BIGINT, 
                        name TEXT, 
                        place_of_birth TEXT, 
                        popularity FLOAT) 
                """ )
    
    connection.execute(query)   
    
    
        
    query = text(""" CREATE TABLE IF NOT EXISTS CREDITS (
                        rol TEXT NOT NULL,
                        imdb_id TEXT NOT NULL,
                        people_id BIGINT NOT NULL,
                        FOREIGN KEY (imdb_id) REFERENCES MOVIES(imdb_id),
                        FOREIGN KEY (people_id) REFERENCES PEOPLE(people_id)
                        )
                 """)
    
    connection.execute(query)  

In [78]:
df_movies.head()

,primaryTitle,isAdult,startYear,runtimeMinutes,genres,averageRating,numVotes,budget,revenue
imdb_id,,,,,,,,,
tt0010323,The Cabinet of Dr. Caligari,0,1920,67,"Horror,Mystery,Thriller",8.0,71863,18000,8811
tt0012349,The Kid,0,1921,68,"Comedy,Drama,Family",8.2,137619,250000,2500000
tt0013442,Nosferatu: A Symphony of Horror,0,1922,94,"Fantasy,Horror",7.8,108889,0,19054
tt0015324,Sherlock Jr.,0,1924,45,"Action,Comedy,Romance",8.2,58967,0,0
tt0015648,Battleship Potemkin,0,1925,66,"Drama,History,Thriller",7.9,62361,0,45100


<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Volcar los datos de los `DataFrame` `df_movies`, `df_people` y `df_credits` a la base de datos creada. Para ello, es necesario utilizar el parámetro `if_exists='append'` en el método `DataFrame.to_sql`.

In [73]:
df_movies.to_sql('MOVIES',engine,if_exists='append')

4265

In [74]:
df_people.to_sql('PEOPLE',engine,if_exists='append')

6406

In [ ]:
df_credits_filtered.to_sql('CREDITS',engine,if_exists='append')

IntegrityError: (sqlite3.IntegrityError) FOREIGN KEY constraint failed
[SQL: INSERT INTO "CREDITS" (imdb_id, rol, people_id) VALUES (?, ?, ?)]
[parameters: [('tt0010323', 'cast', 3000), ('tt0010323', 'cast', 3001), ('tt0010323', 'cast', 590591), ('tt0012349', 'cast', 13848), ('tt0012349', 'cast', 19426), ('tt0012349', 'cast', 63378), ('tt0013442', 'cast', 9839), ('tt0013442', 'cast', 9840)  ... displaying 10 of 17414 total bound parameter sets ...  ('tt9866072', 'director', 61175), ('tt9893250', 'director', 111588)]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [ ]:
credit_ids = set(df_credits_filtered["people_id"].unique())
people_ids = set(df_people["people_id"].unique())

missing_in_people = credit_ids - people_ids 
missing_in_credits = people_ids - credit_ids  

print("IDs in df_credits but missing in df_people:", missing_in_people)
print("IDs in df_people but missing in df_credits:", missing_in_credits)


IDs in df_credits but missing in df_people: set()
IDs in df_people but missing in df_credits: {1365769, 1375501, 25872, 58001, 12178, 1652371, 6295, 86424, 537, 2221, 79405, 1515055, 1442866, 1846, 38582, 585655, 1487546, 187, 9661, 21693, 136127, 1901375, 110916, 54470, 92614, 3664, 11478, 31581, 1898, 84076, 9070, 6385, 63992, 21629}


In [121]:
# Get unique people_id values from df_people
valid_people_ids = set(df_people["people_id"].unique())

# Filter df_credits to remove rows where people_id is missing in df_people
df_credits_filtered = df_credits[df_credits["people_id"].isin(valid_people_ids)]


In [126]:
# Get unique IDs from both dataframes
people_ids = set(df_people["people_id"].unique())
credit_ids = set(df_credits["people_id"].unique())

# Find missing IDs
missing_ids = {'1898', '92614', '63992', '11478', '2221', '1652371', '6295', '3664', 
               '31581', '6385', '58001', '1442866', '12178', '9070', '585655', 
               '1365769', '1901375', '25872', '86424', '54470', '1375501', '136127', 
               '537', '1515055', '187', '1487546', '84076', '110916', '38582', 
               '79405', '1846', '21629', '21693', '9661'}

# Remove rows where people_id is in missing_ids
df_credits_cleaned = df_credits[~df_credits["people_id"].isin(missing_ids)].copy()

# Verify the removal
remaining_ids = set(df_credits_cleaned["people_id"].unique())
new_missing = remaining_ids - people_ids

print("Number of rows in original df_credits:", len(df_credits))
print("Number of rows in cleaned df_credits:", len(df_credits_cleaned))
print("Number of IDs removed:", len(missing_ids))
print("Any remaining missing IDs:", new_missing)

Number of rows in original df_credits: 17414
Number of rows in cleaned df_credits: 17414
Number of IDs removed: 34
Any remaining missing IDs: set()


In [128]:
df_credits_cleaned.to_sql('CREDITS',engine,if_exists='append')

IntegrityError: (sqlite3.IntegrityError) FOREIGN KEY constraint failed
[SQL: INSERT INTO "CREDITS" (imdb_id, rol, people_id) VALUES (?, ?, ?)]
[parameters: [('tt0010323', 'cast', '3000'), ('tt0010323', 'cast', '3001'), ('tt0010323', 'cast', '590591'), ('tt0012349', 'cast', '13848'), ('tt0012349', 'cast', '19426'), ('tt0012349', 'cast', '63378'), ('tt0013442', 'cast', '9839'), ('tt0013442', 'cast', '9840')  ... displaying 10 of 17414 total bound parameter sets ...  ('tt9866072', 'director', '61175'), ('tt9893250', 'director', '111588')]]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

In [ ]:
engine.dispose()

En este punto, se dispone de una pequeña base de datos SQL con los datos extraídos. Esta base de datos, además de forma programática, puede explorarse y manipularse con el programa [_DB Browser for SQLite_](https://sqlitebrowser.org/).

<div align="right">
<a href="#indice"><font size=5 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<a id="section4"></a>
## <font color="#00586D"> 4. Algunas consultas ejemplo</font>
<br>

Llegado este punto, se dispone de un conjunto de datos sobre los que se pueden hacer consultas similares a las que se hicieron en el *Capstone III*. A continuación se propondrán dos a modo de ejemplo (no es el objetivo principal).

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener,  a partir de los `DataFrame`, los datos de los 10 actores o actrices asociados a una mayor recaudación. 

In [ ]:
pd.options.display.max_rows = 10

... # COMPLETAR

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener, mediante una consulta SQL a la base de datos `/data/movies.db` las películas de 2023 que han obtenido una puntuación mayor que 7.5.

In [ ]:
engine = create_engine('sqlite:///data/movies.db')  


pd.read_sql... # COMPLETAR

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener, mediante una consulta SQL a la base de datos `/data/movies.db` las 10 películas con mayor puntuación.

In [ ]:
pd.read_sql...# COMPLETAR

<font size=2 color="#00586D"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#00586D"></i></font> Obtener, mediante una consulta SQL a la base de datos `/data/movies.db` el director de cada película.

In [ ]:
query = ... # COMPLETAR

pd.read_sql(query, engine)

In [ ]:
engine.dispose()

<div><font size=3 color="#00586D"> <i class="fa fa-check-square-o" aria-hidden="true"></i></font></div>

<!--comment -->

<div align="right">
<a href="#indice"><font size=5 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<a id="section5"></a>
## <font color="#00586D"> 5. Conclusiones</font>
<br>

En este trabajo se ha partido de unos conjuntos de datos en formato `.csv`. A partir de ellos, y una vez seleccionados los datos de interés, se ha obtenido información adicional mediante diversas llamadas a APIs, y que ha sido almacenada en una base de datos NoSQL. Una vez disponibles los datos, se ha hecho la transformación correspondiente para seleccionar y organizar los conjuntos de datos de forma que puedan hacerse consultas que, en este caso, han sido sencillas.

En el módulo 13 de CIDaeN se verán procedimientos  en la que estas operaciones se hacen a gran escala, partiendo de *DataLakes* y otras fuentes, y orquestando los procesos de extracción, transformación y carga (ETL) en bases de datos destinadas a la analítica.

<div align="right">
<a href="#indice"><font size=5 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<a id="sectionA"></a>
## <font color="#00586D"> A. Trabajo adicional (opcional)</font>
<br>

Como trabajo adicional, o incluso como base para hacer un TFM (añadiendo algo de visualización y/o modelado), se puede extender este estudio de varias formas. Se proponen dos. 

<a id="sectionA1"></a>
### <font color="#00586D"> A.1. Palabras clave. </font>
<br>

* La llamada `https://api.themoviedb.org/3/movie/'+imdb_id+'/keywords` devuelve las palabras clave correspondientes a la película con identificador `imdb_id`. 

* Se pueden encontrar las keywords de todas las películas de `df_movies` en la colección `cidaendb.keywords`. Para acceder a través de la API es necesario utilizar la API key.

In [ ]:
import os
api_key = os.environ.get('CIDAEN_API_KEY', '<YOUR_CIDAEN_API_KEY>')

<a id="sectionA2"></a>
### <font color="#00586D"> A.2 Datos sobre series y episodios. </font>
<br>

* El archivo  `data/imdb/title.episode.tsv.gz` contiene, para cada episodio, el identificador de la serie a la que pertenece, el número temporada, y el número de episodio (entradas de tipo `tvEpisode` en `data/imdb/title.basic.tsv.gz`).

Esta información puede ser ampliada con los datos de _TMDb_. Para ello, las llamadas más importantes son:


* `https://api.themoviedb.org/3/find/{imdb_id}?external_source=imdb_id` busca la entrada correspondiente al objeto con el correspondiente `imdb_id`. Entre la información que devuelve está el identificador en _TMDb_, necesario para hacer otras llamadas, que no aceptan `imdb_id.  

* `https://api.themoviedb.org/3/tv/{series_id}`, en la que `series_id` es el identificador de la sire en _TMDB_, devuelve información bastante completa sobre una serie. 

* `https://api.themoviedb.org/3/tv/{series_id}/season/{season}/episode/{episode}/external_ids` devuelve información sobre cada capítulo.


In [ ]:
imdb_got = 'tt0944947'  # Juego de Tronos
tmdb_got = 1399


imdb_bob = 'tt4283088'  # Capítulo de la batalla de los bastardos, el mejor de la historia de la televisión.
tmdb_bob = 1187404


<div align="right">
<a href="#indice"><font size=5 color="#00586D"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<a href="#indice"><font size=6 color="#00586D"><i class="fa fa-coffee" aria-hidden="true"></i></font></a>
</div>